##Data prep (CSV → tensors + graph)

In [ ]:
import os, glob
import pandas as pd, numpy as np
import gc

DATA_DIR = r"D:\OneDrive\Desktop\AUST\Thesis_docs"
cands = sorted(glob.glob(f"{DATA_DIR}/*.csv*"), key=os.path.getmtime)
assert len(cands) > 0, "No CSV files found"
PATH = cands[-1]
print("Using:", PATH)

# --- Load & prepare (same as your code, with Drive path + compression='infer') ---
import pandas as pd, numpy as np

df = pd.read_csv(PATH, compression="infer", low_memory=False)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["edge_id"]   = df["u"].astype(str) + "_" + df["v"].astype(str) + "_" + df["key"].astype(str)

# Free-flow time per edge (for later features/TTI)
free = (df.drop_duplicates("edge_id")
          .set_index("edge_id")[["length_m","free_speed_kmh"]])
free["free_time_s"] = free["length_m"] / (free["free_speed_kmh"] * 1000/3600)

# quick sanity peek
print("Rows:", len(df), "Unique edges:", free.shape[0])
display(df.head())
display(free.head())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using: /content/drive/MyDrive/traffic_data/dhakas_timeseries_2025-02-15_7d_15min.csv
Rows: 3360000 Unique edges: 5000


,timestamp,u,v,key,road_type,length_m,free_speed_kmh,traffic_factor,current_speed_kmh,travel_time_seconds,edge_id
0,2025-02-15 00:00:00+06:00,5063403502,5063403483,0,residential,27.835,15.0,0.2469,12.04,8.32,5063403502_5063403483_0
1,2025-02-15 00:00:00+06:00,9908994079,9909890955,0,residential,240.295,15.0,0.1326,13.41,64.52,9908994079_9909890955_0
2,2025-02-15 00:00:00+06:00,387124254,387124256,0,tertiary,81.257,25.0,0.2262,20.48,14.29,387124254_387124256_0
3,2025-02-15 00:00:00+06:00,5608468019,5608468101,0,residential,86.750,15.0,0.4316,9.82,31.80,5608468019_5608468101_0
4,2025-02-15 00:00:00+06:00,10009255308,10009255322,0,primary,432.464,40.0,0.0799,37.44,41.58,10009255308_10009255322_0


,length_m,free_speed_kmh,free_time_s
edge_id,,,
5063403502_5063403483_0,27.835,15.0,6.680400
9908994079_9909890955_0,240.295,15.0,57.670800
387124254_387124256_0,81.257,25.0,11.701008
5608468019_5608468101_0,86.750,15.0,20.820000
10009255308_10009255322_0,432.464,40.0,38.921760


##Build a complete time grid & pivot (edge-as-columns, time-as-rows)

In [2]:
# timestamps already parsed
ts = df["timestamp"]

# tzinfo is either a tz object (e.g., Asia/Dhaka/+06:00) or None
tzinfo = ts.dt.tz

# build the 15-min grid using the same tz (or none if naive)
ti = pd.date_range(start=ts.min(), end=ts.max(), freq="15min", tz=tzinfo)

Y = (df.pivot_table(index="timestamp", columns="edge_id",
                    values="travel_time_seconds")
       .reindex(ti)  # align to the full grid
       .sort_index())

# Fill gaps: forward-fill by time, then fall back to free-flow time (column-wise align)
Y = Y.ffill().fillna(free["free_time_s"]).astype("float32")

print(Y.shape)  # (T, N)


(672, 5000)


##Optional covariates

In [3]:
# Static per edge
X_static = free[["length_m","free_speed_kmh"]].astype("float32")
# One-hot road_type
rt = (df.drop_duplicates("edge_id").set_index("edge_id")["road_type"])
X_static = X_static.join(pd.get_dummies(rt, prefix="rt", dtype="float32")).astype("float32")

# Temporal features per time step (shared across edges)
t = Y.index
X_time = pd.DataFrame({
    "hour_sin":  np.sin(2*np.pi*t.hour/24),
    "hour_cos":  np.cos(2*np.pi*t.hour/24),
    "dow_sin":   np.sin(2*np.pi*t.dayofweek/7),
    "dow_cos":   np.cos(2*np.pi*t.dayofweek/7),
    "is_weekend": ((t.dayofweek.isin([4,5])).astype("float32"))  # BD weekend Fri(4), Sat(5)
}, index=t).astype("float32")


##Build the learning graph (edge-centric)

In [4]:
!pip install osmnx networkx numpy torch

import osmnx as ox, networkx as nx, numpy as np, torch

# Load the same graph you used to simulate (save it once as GraphML during data gen).
# Example: download a small Dhaka road network and save it
G_small = ox.graph_from_place("Dhaka, Bangladesh", network_type="drive")
ox.save_graphml(G_small, f"{DATA_DIR}/dhaka_small.graphml")

# Example:
# ox.save_graphml(G_small, "/content/dhaka_small.graphml")
DATA_DIR = "/content/drive/MyDrive/traffic_data"
G = ox.load_graphml(f"{DATA_DIR}/dhaka_small.graphml")

# Map original edges -> edge_id strings used above
def eid(u,v,k): return f"{u}_{v}_{k}"
edge_list = [(u,v,k) for u,v,k in G.edges(keys=True)]
kept = [eid(u,v,k) for (u,v,k) in edge_list if eid(u,v,k) in Y.columns]
edge_to_idx = {e:i for i,e in enumerate(kept)}
# Reorder Y/X_static to match graph order
Y = Y[kept]
X_static = X_static.reindex(kept).fillna(0)

# Line graph adjacency
L = nx.line_graph(G)                 # nodes are (u,v,k)
pairs = []
for e1, e2 in L.edges():
    a, b = eid(*e1), eid(*e2)
    if a in edge_to_idx and b in edge_to_idx:
        pairs.append((edge_to_idx[a], edge_to_idx[b]))
edge_index = torch.tensor(np.array(pairs).T, dtype=torch.long)  # [2, E_L]


/usr/local/lib/python3.12/dist-packages/osmnx/_overpass.py:271: UserWarning: This area is 11 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


##Normalize and make sliding windows

In [5]:
# Normalize target per-edge (robust z)
y_mean = Y.mean(axis=0).astype("float32")
y_std  = Y.std(axis=0).replace(0, 1).astype("float32")
Yz = ((Y - y_mean)/y_std).astype("float32")

# Temporal split (no leakage)
T = len(Y)
t_train, t_val = int(0.7*T), int(0.85*T)  # 70/15/15
Ytr, Yva, Yte = Yz.iloc[:t_train], Yz.iloc[t_train:t_val], Yz.iloc[t_val:]

L = 12   # lookback = 3 hours
H = 4    # horizon  = 1 hour
def make_windows(Yblock):
    X_seq, Y_seq = [], []
    Yvals = Yblock.to_numpy()  # [T', N]
    for t0 in range(L, len(Yvals)-H+1):
        X_seq.append(Yvals[t0-L:t0, :])      # [L, N]
        Y_seq.append(Yvals[t0:t0+H, :])      # [H, N]
    return np.stack(X_seq), np.stack(Y_seq)  # [B, L, N], [B, H, N]

Xtr, Ytr_out = make_windows(Ytr)
Xva, Yva_out = make_windows(Yva)
Xte, Yte_out = make_windows(Yte)


##ST-GCN (TemporalConv → GraphConv → TemporalConv)

In [6]:
!pip -q install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1
!pip -q install -f https://data.pyg.org/whl/torch-2.3.1+cpu.html \
  pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv
!pip -q install torch_geometric==2.5.3


In [7]:
import torch, torch.nn as nn

class SimpleGCNConv(nn.Module):
    def __init__(self, in_dim, out_dim, bias=True):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim, bias=bias)

    def forward(self, x, edge_index, num_nodes=None):
        # x: [B*L, N, C] or [N, C]; edge_index: [2, E]
        if x.dim()==2:
            x = x.unsqueeze(0)  # [1,N,C]
        B, N, C = x.shape
        if num_nodes is None: num_nodes = N

        idx = edge_index
        vals = torch.ones(idx.size(1), device=x.device)

        A = torch.sparse_coo_tensor(idx, vals, size=(N, N), device=x.device)
        loop_idx = torch.arange(N, device=x.device).repeat(2,1)
        I = torch.sparse_coo_tensor(loop_idx, torch.ones(N, device=x.device), (N,N), device=x.device)
        A_hat = (A + I).coalesce()

        deg = torch.sparse.sum(A_hat, dim=1).to_dense().clamp(min=1.0)
        D_inv_sqrt = torch.diag(deg.pow(-0.5))
        A_norm = D_inv_sqrt @ A_hat.to_dense() @ D_inv_sqrt

        x = self.lin(x)                      # [B,N,out]
        out = torch.einsum("ij,bjk->bik", A_norm, x)
        return out.squeeze(0)


python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
gpu available: True
colab cuda runtime (if any): 12.6


In [8]:
import torch
import torch.nn as nn

# ---- Pure-PyTorch GCN layer (CPU-friendly) ----
class SimpleGCNConv(nn.Module):
    """
    A minimal GCN layer using normalized dense adjacency:
        A_norm = D^{-1/2} (A + I) D^{-1/2}
        out    = A_norm @ (X W)
    Shapes:
        X:  [B, N, C_in]  or [N, C_in]
        out:[B, N, C_out] or [N, C_out]
    edge_index: [2, E] (0-based indices)
    Note: Builds a dense N×N A_norm each forward; fine for moderate N.
    """
    def __init__(self, in_dim, out_dim, bias=True):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim, bias=bias)

    def forward(self, x, edge_index, num_nodes=None):
        # Normalize input shape: [B, N, C]
        unsqueeze = False
        if x.dim() == 2:        # [N, C]
            x = x.unsqueeze(0)  # [1, N, C]
            unsqueeze = True
        B, N, _ = x.shape
        if num_nodes is None:
            num_nodes = N

        # Sparse adjacency with self-loops
        idx = edge_index
        vals = torch.ones(idx.size(1), device=x.device, dtype=x.dtype)
        A = torch.sparse_coo_tensor(idx, vals, size=(num_nodes, num_nodes), device=x.device)
        loop_idx = torch.arange(num_nodes, device=x.device)
        I = torch.sparse_coo_tensor(
            torch.stack([loop_idx, loop_idx]),
            torch.ones(num_nodes, device=x.device, dtype=x.dtype),
            size=(num_nodes, num_nodes),
            device=x.device
        )
        A_hat = (A + I).coalesce()

        # D^{-1/2} A_hat D^{-1/2}  (dense for simplicity)
        deg = torch.sparse.sum(A_hat, dim=1).to_dense().clamp(min=1.0)
        D_inv_sqrt = torch.diag(deg.pow(-0.5))
        A_norm = D_inv_sqrt @ A_hat.to_dense() @ D_inv_sqrt   # [N, N]

        # Apply linear, then aggregate
        x_lin = self.lin(x)                                    # [B, N, C_out]
        out = torch.einsum("ij,bjk->bik", A_norm, x_lin)       # [B, N, C_out]

        return out.squeeze(0) if unsqueeze else out


# ---- Your ST-GCN (now using SimpleGCNConv) ----
class TemporalBlock(nn.Module):
    def __init__(self, cin, cout, k=3, d=1):
        super().__init__()
        self.conv = nn.Conv2d(cin, 2*cout, (k,1), dilation=(d,1),
                              padding=(d*(k-1),0))
        self.glu  = nn.GLU(dim=1)
        self.bn   = nn.BatchNorm2d(cout)
    def forward(self, x):                # x: [B,C,L,N]
        x = self.conv(x)
        x = self.glu(x)
        return self.bn(x)

class STBlock(nn.Module):
    def __init__(self, c_in, c_mid):
        super().__init__()
        self.t1 = TemporalBlock(c_in, c_mid)
        self.gc = SimpleGCNConv(c_mid, c_mid)   # <--- replaced GCNConv
        self.t2 = TemporalBlock(c_mid, c_mid)
    def forward(self, x, edge_index):           # x: [B,C,L,N]
        x = self.t1(x)                           # [B,Cm,L,N]
        B, C, Lt, N = x.shape
        # reshape to feed graph conv over nodes at each time step
        xg = x.transpose(1,2).reshape(B*Lt, C, N).transpose(1,2)   # [B*L, N, C]
        xg = self.gc(xg, edge_index)                                # [B*L, N, C]
        x = xg.transpose(1,2).reshape(B, Lt, C, N).transpose(1,2)   # [B,C,L,N]
        x = self.t2(x)
        return x

class STGCN(nn.Module):
    def __init__(self, lookback, horizon, n_nodes, hidden=32):
        super().__init__()
        self.enc1 = STBlock(1, hidden)
        self.enc2 = STBlock(hidden, hidden)
        self.head = nn.Conv2d(hidden, horizon, (1,1))
    def forward(self, x, edge_index):           # x: [B,1,L,N]
        x = self.enc1(x, edge_index)
        x = self.enc2(x, edge_index)
        y = self.head(x)                         # [B,H,L,N]
        return y[:, :, -1, :]                    # [B,H,N]


In [9]:
import torch
import torch.nn as nn
import numpy as np

device = torch.device("cpu")

# ---- Pure-PyTorch GCN layer (no PyG) ----
class SimpleGCNConv(nn.Module):
    """
    Minimal GCN: A_norm @ (X W) with A_norm = D^{-1/2} (A+I) D^{-1/2}
    x: [B, N, C]  or [N, C]; edge_index: [2, E] (0-based)
    Builds dense A_norm each forward (fine for moderate N).
    """
    def __init__(self, in_dim, out_dim, bias=True):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim, bias=bias)

    def forward(self, x, edge_index, num_nodes=None):
        squeeze_back = False
        if x.dim() == 2:          # [N,C]
            x = x.unsqueeze(0)    # [1,N,C]
            squeeze_back = True
        B, N, _ = x.shape
        if num_nodes is None: num_nodes = N

        idx = edge_index.to(x.device)
        vals = torch.ones(idx.size(1), device=x.device, dtype=x.dtype)
        A = torch.sparse_coo_tensor(idx, vals, size=(num_nodes, num_nodes), device=x.device)
        loop = torch.arange(num_nodes, device=x.device)
        I = torch.sparse_coo_tensor(torch.stack([loop, loop]),
                                    torch.ones(num_nodes, device=x.device, dtype=x.dtype),
                                    size=(num_nodes, num_nodes), device=x.device)
        A_hat = (A + I).coalesce()

        deg = torch.sparse.sum(A_hat, dim=1).to_dense().clamp(min=1.0)
        D_inv_sqrt = torch.diag(deg.pow(-0.5))
        A_norm = D_inv_sqrt @ A_hat.to_dense() @ D_inv_sqrt  # [N,N]

        x_lin = self.lin(x)                                  # [B,N,C_out]
        out = torch.einsum("ij,bjk->bik", A_norm, x_lin)     # [B,N,C_out]
        return out.squeeze(0) if squeeze_back else out


# ---- ST-Transformer (temporal attention + light graph mixing) ----
class STTransformerCPU(nn.Module):
    """
    Input:  x  [B, 1, L, N]   (single channel: normalized travel_time)
    Output: y  [B, H, N]
    """
    def __init__(self, lookback, horizon, n_nodes, d_model=64, nhead=4, nlayers=2):
        super().__init__()
        self.L = lookback
        self.H = horizon
        self.N = n_nodes

        self.inp = nn.Linear(1, d_model)  # per-node per-time scalar -> d_model
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4*d_model,
            batch_first=True, dropout=0.1, activation='gelu'
        )
        self.enc = nn.TransformerEncoder(enc_layer, nlayers)
        self.gmix = SimpleGCNConv(d_model, d_model)        # light graph mixing
        self.head = nn.Linear(d_model, horizon)            # per-node horizon

    def forward(self, x, edge_index):                      # x: [B,1,L,N]
        B, C, L, N = x.shape
        assert C == 1 and L == self.L and N == self.N

        # reshape per-node sequences: (B*N, L, 1) -> Transformer
        x_seq = x.permute(0, 3, 2, 1).reshape(B*N, L, 1)   # (B*N, L, 1)
        z = self.inp(x_seq)                                # (B*N, L, d)
        z = self.enc(z)                                    # temporal attention

        # take last time step’s embedding per node
        z_last = z[:, -1, :]                               # (B*N, d)
        z_last = z_last.view(B, N, -1)                     # (B, N, d)

        # graph mixing once at the end of the encoder
        z_g = self.gmix(z_last, edge_index)                # (B, N, d)

        # per-node horizon prediction
        y = self.head(z_g)                                 # (B, N, H)
        return y.permute(0, 2, 1)                          # [B, H, N]


In [10]:
# --- assumes df with columns: timestamp, u, v, key, travel_time_seconds, length_m, free_speed_kmh
import pandas as pd, numpy as np

# 1) Ensure timestamp and edge_id
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["edge_id"] = df["u"].astype(str)+"_"+df["v"].astype(str)+"_"+df["key"].astype(str)

# 2) Free-flow time per edge (for gap fill)
free = (df.drop_duplicates("edge_id")
          .set_index("edge_id")[["length_m","free_speed_kmh"]])
free["free_time_s"] = free["length_m"] / (free["free_speed_kmh"] * 1000/3600)

# 3) Build full 15-min grid with same tz (or naive if none)
ts = df["timestamp"]
tzinfo = ts.dt.tz
ti = pd.date_range(start=ts.min(), end=ts.max(), freq="15min", tz=tzinfo)

# 4) Pivot to Y[T×N] with columns in a fixed edge order
Y = (df.pivot_table(index="timestamp", columns="edge_id",
                    values="travel_time_seconds")
       .reindex(ti).sort_index())

# 5) Fill tiny gaps: forward fill then free-flow fallback
Y = Y.ffill().fillna(free["free_time_s"]).astype("float32")

# 6) Save column order (edge list) for later alignment
edge_cols = Y.columns.tolist()
N = len(edge_cols)
T = len(Y)

# 7) Normalize per edge
y_mean = Y.mean(axis=0).astype("float32")
y_std  = Y.std(axis=0).replace(0, 1).astype("float32")
Yz = ((Y - y_mean)/y_std).astype("float32")

# 8) Temporal split (70/15/15)
t_train, t_val = int(0.70*T), int(0.85*T)
Ytr, Yva, Yte = Yz.iloc[:t_train], Yz.iloc[t_train:t_val], Yz.iloc[t_val:]

# 9) Sliding windows
L = 12   # lookback (3 hours)
H = 4    # horizon  (1 hour)

def make_windows(Yblock, L, H):
    arr = Yblock.to_numpy()             # [T', N]
    X, Yout = [], []
    for t0 in range(L, len(arr) - H + 1):
        X.append(arr[t0-L:t0, :])       # [L, N]
        Yout.append(arr[t0:t0+H, :])    # [H, N]
    return np.stack(X), np.stack(Yout)  # [B, L, N], [B, H, N]

Xtr, Ytr_out = make_windows(Ytr, L, H)
Xva, Yva_out = make_windows(Yva, L, H)
Xte, Yte_out = make_windows(Yte, L, H)

print("Shapes ->",
      "Xtr", Xtr.shape, "Ytr_out", Ytr_out.shape,
      "Xva", Xva.shape, "Yva_out", Yva_out.shape,
      "Xte", Xte.shape, "Yte_out", Yte_out.shape,
      "| N =", N)


Shapes -> Xtr (455, 12, 5000) Ytr_out (455, 4, 5000) Xva (86, 12, 5000) Yva_out (86, 4, 5000) Xte (86, 12, 5000) Yte_out (86, 4, 5000) | N = 5000


In [11]:
import torch

def precompute_A_norm(edge_index, N, device="cpu", dtype=torch.float32):
    """
    Returns A_norm (D^{-1/2}(A+I)D^{-1/2}) as sparse COO; no dense N×N in forward.
    If edge_index is empty, this reduces to identity.
    """
    edge_index = edge_index.to(device)
    if edge_index.numel() == 0:
        # Identity (self-loops only)
        idx = torch.arange(N, device=device)
        values = torch.ones(N, dtype=dtype, device=device)
        return torch.sparse_coo_tensor(
            torch.stack([idx, idx]), values, size=(N, N), device=device
        ).coalesce()

    vals = torch.ones(edge_index.size(1), dtype=dtype, device=device)
    A = torch.sparse_coo_tensor(edge_index, vals, (N, N), device=device).coalesce()
    # add self-loops
    idx = torch.arange(N, device=device)
    I = torch.sparse_coo_tensor(
        torch.stack([idx, idx]), torch.ones(N, dtype=dtype, device=device),
        (N, N), device=device
    ).coalesce()
    A_hat = (A + I).coalesce()

    deg = torch.sparse.sum(A_hat, dim=1).to_dense().clamp(min=1.0)  # [N]
    d_inv_sqrt = deg.pow(-0.5)
    r, c = A_hat.indices()
    v = A_hat.values() * d_inv_sqrt[r] * d_inv_sqrt[c]
    A_norm = torch.sparse_coo_tensor(
        torch.stack([r, c]), v, (N, N), device=device
    ).coalesce()
    return A_norm


In [12]:
import torch
import torch.nn as nn

class SimpleGCNConvCached(nn.Module):
    """
    Uses precomputed sparse A_norm. No dense N×N allocation in forward.
    x: [B, N, C_in] or [N, C_in]
    """
    def __init__(self, in_dim, out_dim, A_norm: torch.Tensor, bias=True):
        super().__init__()
        assert A_norm.is_sparse
        self.lin = nn.Linear(in_dim, out_dim, bias=bias)
        self.N = A_norm.size(0)
        # Store indices/values as buffers for fast rebuild on forward device
        self.register_buffer("A_idx", A_norm.indices())
        self.register_buffer("A_val", A_norm.values())

    def forward(self, x):
        squeeze_back = False
        if x.dim() == 2:              # [N, C]
            x = x.unsqueeze(0)        # [1, N, C]
            squeeze_back = True
        B, N, _ = x.shape
        assert N == self.N
        # Rebuild sparse tensor on x.device
        A_norm = torch.sparse_coo_tensor(self.A_idx, self.A_val, (N, N), device=x.device)
        x_lin = self.lin(x)           # [B, N, Cout]
        # torch.sparse.mm is 2D; loop over batch (B is small on CPU)
        outs = []
        for b in range(B):
            outs.append(torch.sparse.mm(A_norm, x_lin[b]))  # [N, Cout]
        out = torch.stack(outs, 0)     # [B, N, Cout]
        return out.squeeze(0) if squeeze_back else out


In [13]:
import torch.nn as nn

class STTransformerCPU(nn.Module):
    """
    Input:  x [B, 1, L, N]
    Output: y [B, H, N]
    """
    def __init__(self, lookback, horizon, n_nodes, A_norm_sparse,
                 d_model=32, nhead=2, nlayers=1):
        super().__init__()
        self.L, self.H, self.N = lookback, horizon, n_nodes

        self.inp = nn.Linear(1, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4*d_model,
            batch_first=True, dropout=0.1, activation="gelu"
        )
        self.enc = nn.TransformerEncoder(enc_layer, nlayers)
        self.gmix = SimpleGCNConvCached(d_model, d_model, A_norm_sparse)  # <-- cached sparse
        self.head = nn.Linear(d_model, horizon)

    def forward(self, x):                                # x: [B,1,L,N]
        B, C, L, N = x.shape
        assert C == 1 and L == self.L and N == self.N

        x_seq = x.permute(0, 3, 2, 1).reshape(B*N, L, 1) # (B*N, L, 1)
        z = self.inp(x_seq)                              # (B*N, L, d)
        z = self.enc(z)                                  # temporal attention
        z_last = z[:, -1, :].view(B, N, -1)              # (B,N,d)

        z_g = self.gmix(z_last)                          # (B,N,d), sparse mm
        y = self.head(z_g)                               # (B,N,H)
        return y.permute(0, 2, 1)                        # [B,H,N]


In [14]:
import torch, numpy as np, gc
device = torch.device("cpu")

# Free memory from big intermediates you no longer need:
# del df ; gc.collect()

# Build cached sparse A_norm once
N = Xtr.shape[2]
edge_index_t = edge_index.long() if torch.is_tensor(edge_index) else torch.tensor(edge_index, dtype=torch.long)
A_norm_sparse = precompute_A_norm(edge_index_t, N, device="cpu", dtype=torch.float32)

# Model + optim
model = STTransformerCPU(lookback=L, horizon=H, n_nodes=N, A_norm_sparse=A_norm_sparse,
                         d_model=32, nhead=2, nlayers=1).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
loss_fn = torch.nn.L1Loss()

# Dataloaders built earlier; reduce batch size to limit RAM:
from torch.utils.data import DataLoader, TensorDataset

def to_torch_cpu(X, Y):
    X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # [B,1,L,N]
    Y = torch.tensor(Y, dtype=torch.float32)               # [B,H,N]
    return X, Y

Xtr_t, Ytr_t = to_torch_cpu(Xtr, Ytr_out)
Xva_t, Yva_t = to_torch_cpu(Xva, Yva_out)

train_loader = DataLoader(TensorDataset(Xtr_t, Ytr_t), batch_size=16, shuffle=True)
val_loader   = DataLoader(TensorDataset(Xva_t, Yva_t), batch_size=16, shuffle=False)

# (Optional) throttle CPU threads to reduce memory spikes
import torch
torch.set_num_threads(4)

def run_epoch(loader, train=True):
    model.train(train)
    total = 0.0; count = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        with torch.set_grad_enabled(train):
            pred = model(xb)                 # no edge_index arg now
            loss = loss_fn(pred, yb)
        if train:
            opt.zero_grad(); loss.backward(); opt.step()
        bs = xb.size(0); total += loss.item()*bs; count += bs
    return total / max(count, 1)

best_val = float("inf"); best_state=None; patience=5; bad=0
for ep in range(50):
    tr = run_epoch(train_loader, True)
    va = run_epoch(val_loader, False)
    print(f"ep{ep:02d}  train_MAE:{tr:.4f}  val_MAE:{va:.4f}")
    if va < best_val - 1e-4:
        best_val = va
        best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
        bad = 0
    else:
        bad += 1
        if bad >= patience:
            print("Early stopping."); break

if best_state is not None:
    model.load_state_dict(best_state)


ep00  train_MAE:0.6123  val_MAE:0.4597
ep01  train_MAE:0.4500  val_MAE:0.4349
ep02  train_MAE:0.4416  val_MAE:0.4422
ep03  train_MAE:0.4373  val_MAE:0.4309
ep04  train_MAE:0.4339  val_MAE:0.4223
ep05  train_MAE:0.4345  val_MAE:0.4283
ep06  train_MAE:0.4331  val_MAE:0.4232
ep07  train_MAE:0.4339  val_MAE:0.4281
ep08  train_MAE:0.4305  val_MAE:0.4243
ep09  train_MAE:0.4290  val_MAE:0.4221
ep10  train_MAE:0.4288  val_MAE:0.4220
ep11  train_MAE:0.4271  val_MAE:0.4223
ep12  train_MAE:0.4273  val_MAE:0.4188
ep13  train_MAE:0.4288  val_MAE:0.4321
ep14  train_MAE:0.4277  val_MAE:0.4231
ep15  train_MAE:0.4263  val_MAE:0.4201
ep16  train_MAE:0.4254  val_MAE:0.4194
ep17  train_MAE:0.4265  val_MAE:0.4202
Early stopping.
